# Day 3 — Tasks and Plugins: Shell, Python, HTTP, and More

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-03-tasks-plugins.ipynb#scrollTo=a1b2c3d4)

**Course:** Kestra for Data Engineers  
**Badge:** Learn  

Kestra ships with hundreds of plugins — each one is a Java class your YAML flow can invoke. Today you'll explore the core plugin categories, build flows for Shell, HTTP, and Python tasks, and learn how task outputs pass data downstream.

**By the end of this notebook you will:**
- Navigate the Kestra plugin library and understand the naming convention
- Generate YAML for `Log`, `Shell`, `HTTP`, and `Python` tasks using PyYAML
- Understand how `outputs.taskId.vars` connects tasks in a pipeline
- Write a multi-task flow that fetches an API, processes the response, and logs a summary

In [ ]:
%pip install -q pyyaml requests pandas

## 1. The Plugin Library — Naming Convention

Every Kestra plugin type follows the Java package convention:

```
io.kestra.plugin.<package>.<ClassName>
```

| Package | Examples |
|---------|----------|
| `core.log` | `Log` — emit structured log messages |
| `core.http` | `Request`, `Download` — call REST APIs |
| `scripts.shell` | `Commands` — run Bash in a container |
| `scripts.python` | `Commands` — run Python in a virtualenv |
| `jdbc.postgresql` | `Query`, `Queries` — SQL over JDBC |
| `aws.s3` | `Upload`, `Download` — S3 object operations |

Browse the full catalogue at [kestra.io/plugins](https://kestra.io/plugins/).

In [ ]:
import yaml

# Plugin naming convention illustrated
plugin_examples = [
    {"type": "io.kestra.plugin.core.log.Log", "category": "Core", "purpose": "Emit a log message"},
    {"type": "io.kestra.plugin.core.http.Request", "category": "HTTP", "purpose": "Call a REST API"},
    {"type": "io.kestra.plugin.scripts.shell.Commands", "category": "Shell", "purpose": "Run Bash commands"},
    {"type": "io.kestra.plugin.scripts.python.Commands", "category": "Python", "purpose": "Run Python script"},
]

print(f"{'Plugin Type':<50} {'Category':<10} {'Purpose'}")
print("-" * 80)
for p in plugin_examples:
    print(f"{p['type']:<50} {p['category']:<10} {p['purpose']}")

> **Key insight:** The full type string is the only identifier Kestra needs. No imports, no dependencies in your flow YAML — Kestra downloads and caches the plugin JAR at startup.

## 2. The Log Task — Structured Messages

`io.kestra.plugin.core.log.Log` is the simplest task. It emits a message to the execution log at a specified level. Use it to:
- Confirm a flow branch was reached
- Print runtime values like `{{ inputs.env }}`
- Debug output from previous tasks

**Properties:** `message` (required), `level` (INFO / WARN / ERROR, default INFO)

In [ ]:
import yaml

log_flow = {
    "id": "hello-log",
    "namespace": "tutorial.day03",
    "description": "Demonstrates structured log messages at different levels",
    "tasks": [
        {
            "id": "info_message",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Pipeline started — environment: {{ inputs.env | default('dev') }}",
            "level": "INFO"
        },
        {
            "id": "warn_message",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Skipping optional step: no data file provided",
            "level": "WARN"
        },
        {
            "id": "debug_value",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Execution ID: {{ execution.id }} — Flow: {{ flow.id }}",
            "level": "INFO"
        }
    ]
}

print(yaml.dump(log_flow, default_flow_style=False, sort_keys=False))

> **Pebble templates** like `{{ inputs.env | default('dev') }}` are evaluated at runtime by the Kestra executor. The `| default(...)` filter supplies a fallback if the value is absent.

## 3. Shell Commands — Running Bash in a Container

`io.kestra.plugin.scripts.shell.Commands` runs one or more shell commands inside a Docker container (or on the worker host for the PROCESS runner).

**Key properties:**
- `commands` — list of shell strings to execute in sequence
- `runner` — `DOCKER` (isolated) or `PROCESS` (worker host)
- `docker.image` — the container image (default: `ubuntu:latest`)
- `outputFiles` — glob patterns of files to capture as Kestra internal storage URIs

In [ ]:
shell_flow = {
    "id": "shell-demo",
    "namespace": "tutorial.day03",
    "description": "Fetch public data with curl and produce a summary file",
    "tasks": [
        {
            "id": "fetch_and_summarise",
            "type": "io.kestra.plugin.scripts.shell.Commands",
            "runner": "DOCKER",
            "docker": {"image": "curlimages/curl:latest"},
            "commands": [
                "curl -s 'https://jsonplaceholder.typicode.com/posts?_limit=5' -o posts.json",
                "echo 'Fetched posts count:'",
                "cat posts.json | python3 -c \"import json,sys; data=json.load(sys.stdin); print(len(data))\"",
                "echo 'First post title:'",
                "cat posts.json | python3 -c \"import json,sys; d=json.load(sys.stdin); print(d[0]['title'])\""
            ],
            "outputFiles": ["*.json"]
        }
    ]
}

print(yaml.dump(shell_flow, default_flow_style=False, sort_keys=False))

> `outputFiles: ['*.json']` captures all JSON files written during the task and stores them in Kestra's internal S3-compatible object store. Downstream tasks reference them via `{{ outputs.fetch_and_summarise.outputFiles['posts.json'] }}`.

## 4. HTTP Request — Calling REST APIs

`io.kestra.plugin.core.http.Request` makes an HTTP call and stores the response body and status code as task outputs — no shell needed.

**Key properties:**
- `uri` — the endpoint URL (supports Pebble templates)
- `method` — GET / POST / PUT / DELETE
- `headers` — map of request headers
- `body` — request body string (for POST/PUT)

**Outputs:** `body` (string), `code` (HTTP status int), `headers` (map)

In [ ]:
http_flow = {
    "id": "http-request-demo",
    "namespace": "tutorial.day03",
    "description": "Fetch a public JSON API and log the response status",
    "tasks": [
        {
            "id": "fetch_posts",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/posts/1",
            "method": "GET",
            "headers": {"Accept": "application/json"}
        },
        {
            "id": "log_response",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "HTTP {{ outputs.fetch_posts.code }} — body: {{ outputs.fetch_posts.body | truncate(120) }}"
        }
    ]
}

print(yaml.dump(http_flow, default_flow_style=False, sort_keys=False))

# Simulate what the HTTP response would look like
import requests, json
resp = requests.get("https://jsonplaceholder.typicode.com/posts/1")
print(f"\n# Live simulation — HTTP {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

> `outputs.fetch_posts.body` is a plain string containing the response body. Use Pebble's `| truncate(N)` filter to cap log length, or pipe it to a Python task for structured parsing.

## 5. Python Script Task — Pandas in a Managed Virtualenv

`io.kestra.plugin.scripts.python.Commands` runs Python code inside an auto-managed virtualenv. Declare pip packages in `beforeCommands` and Kestra caches the environment between runs.

**Key properties:**
- `script` — inline Python source (multiline string)
- `beforeCommands` — shell commands run before the script (e.g., `pip install pandas`)
- `inputFiles` — map of filename → content or URI to inject into the working directory
- `outputFiles` — glob patterns to capture

In [ ]:
python_flow = {
    "id": "python-pandas-demo",
    "namespace": "tutorial.day03",
    "description": "Run a Python task that reads public CSV data and prints statistics",
    "tasks": [
        {
            "id": "analyse_data",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "script": (
                "import pandas as pd\n"
                "url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'\n"
                "df = pd.read_csv(url)\n"
                "print(f'Shape: {df.shape}')\n"
                "print(f'Columns: {list(df.columns)}')\n"
                "print(df.describe().to_string())\n"
            )
        }
    ]
}

print(yaml.dump(python_flow, default_flow_style=False, sort_keys=False))

# Simulate locally
import pandas as pd
url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'
df = pd.read_csv(url)
print(f"\n# Live simulation")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head(3).to_string())

> Kestra hashes the `beforeCommands` list and caches the resulting virtualenv. Re-running the same flow skips pip install — execution is much faster after the first run.

## 6. Task Outputs — Passing Data Between Tasks

Every Kestra task can expose named output variables. Reference them in downstream tasks with:

```
{{ outputs.<taskId>.<outputName> }}
```

For Python tasks using `Kestra.outputs({...})`, outputs are available under `outputs.<taskId>.vars`:

```
{{ outputs.analyse.vars.row_count }}
```

For HTTP tasks:
- `outputs.<taskId>.body` — response body string
- `outputs.<taskId>.code` — HTTP status code

For Shell/Python tasks:
- `outputs.<taskId>.outputFiles` — map of captured file URIs

In [ ]:
# Multi-task flow: HTTP fetch → Python analyse → Log summary
pipeline_flow = {
    "id": "fetch-analyse-log",
    "namespace": "tutorial.day03",
    "description": "Chain HTTP → Python → Log using task output variables",
    "tasks": [
        {
            "id": "fetch_users",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/users",
            "method": "GET"
        },
        {
            "id": "count_users",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": [],
            "script": (
                "import json\n"
                "# In Kestra, the upstream body is injected via Pebble into the script text\n"
                "# Here we use an env variable pattern for illustration\n"
                "import os\n"
                "body = os.environ.get('USERS_BODY', '[]')\n"
                "users = json.loads(body)\n"
                "count = len(users)\n"
                "cities = list({u['address']['city'] for u in users})\n"
                "# Kestra.outputs() publishes named values to downstream tasks\n"
                "Kestra.outputs({'count': count, 'cities': cities})\n"
            ),
            "env": {"USERS_BODY": "{{ outputs.fetch_users.body }}"}
        },
        {
            "id": "log_summary",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Found {{ outputs.count_users.vars.count }} users in {{ outputs.count_users.vars.cities | length }} cities"
        }
    ]
}

print(yaml.dump(pipeline_flow, default_flow_style=False, sort_keys=False))

> `Kestra.outputs({...})` is injected by the Kestra runtime — no import required. It serialises the dict to an internal log line that the executor reads back as output variables.

## 7. Simulate the Output Variable System Locally

In [ ]:
import requests, json

# Step 1: simulate fetch_users
resp = requests.get("https://jsonplaceholder.typicode.com/users")
users = resp.json()
print(f"HTTP {resp.status_code} — {len(users)} users fetched")

# Step 2: simulate count_users
count = len(users)
cities = list({u["address"]["city"] for u in users})
outputs_count_users = {"count": count, "cities": cities}
print(f"\nKestra.outputs: {json.dumps(outputs_count_users, indent=2)}")

# Step 3: simulate log_summary using output vars
log_msg = f"Found {outputs_count_users['count']} users in {len(outputs_count_users['cities'])} cities"
print(f"\nLog message: {log_msg}")
print(f"Cities: {sorted(outputs_count_users['cities'])}")

## Challenge

Build a 3-task Kestra YAML flow that:
1. Fetches all posts from `https://jsonplaceholder.typicode.com/posts` using `io.kestra.plugin.core.http.Request`
2. Runs a Python task that parses the JSON body, counts posts per userId (1–10), and calls `Kestra.outputs({'by_user': counts_dict})`
3. Logs: `"User 1 wrote {{ outputs.count_by_user.vars.by_user['1'] }} posts"`

Bonus: Add a 4th task that uses `io.kestra.plugin.scripts.shell.Commands` to write a CSV summary file and capture it with `outputFiles`.

In [ ]:
# Your solution here
challenge_flow = {
    "id": "posts-summary",
    "namespace": "tutorial.day03.challenge",
    "tasks": [
        # Task 1: fetch posts
        # Task 2: count by userId → Kestra.outputs
        # Task 3: log the count for user 1
    ]
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))

## Recap

| Plugin | Type string | Key output |
|--------|-------------|------------|
| Log | `io.kestra.plugin.core.log.Log` | (none — side-effect only) |
| HTTP | `io.kestra.plugin.core.http.Request` | `outputs.id.body`, `outputs.id.code` |
| Shell | `io.kestra.plugin.scripts.shell.Commands` | `outputs.id.outputFiles` |
| Python | `io.kestra.plugin.scripts.python.Commands` | `outputs.id.vars.*` via `Kestra.outputs()` |

**Tip:** Kestra manages Python virtualenvs automatically per task — no pip install needed in the flow itself. Declare dependencies in `beforeCommands` and Kestra caches them between executions.

**Tomorrow — Day 4:** Triggers and Scheduling — cron expressions, webhook triggers, and flow-to-flow event chains.